# SWYNEX Task 2 – Exploratory Data Analysis (EDA)

**Tools:** Python, Pandas, Plotly & Google Colab  
**Dataset:** Online Retail II – Year 2009–2010  
**Related task:** Uses the same dataset and preparation rules as Task 1.

## Project Overview

This project explores the Online Retail II dataset used in Task 1 to identify sales trends, leading countries, popular products, and customer patterns using Python, Pandas, and interactive Plotly visualizations. The preparation steps are repeated in this notebook so Task 2 can run independently.

## Dataset Information

- **Dataset:** Online Retail II
- **Period used:** Year 2009–2010 sheet
- **Source:** UCI Machine Learning Repository
- **Preparation:** Remove duplicate rows, remove rows with missing Description, use `0` as a placeholder for missing Customer ID, convert Customer ID to integer, and trim Country text.
- **Note:** Customer ID `0` represents a missing ID and is excluded from unique-customer analysis.

## Preparation and EDA Steps

1. Load the selected dataset and prepare it for analysis.
2. Inspect data types and summary statistics.
3. Calculate line-level sales as `Quantity × Price`.
4. Analyse country-wise and monthly net sales.
5. Visualize the top 10 countries by net sales.
6. Compare customer counts across countries, excluding missing-ID placeholders.
7. Identify top-selling products and explore quantity-versus-sales patterns.
8. Summarize the key findings.

**Interpretation note:** Negative quantities and prices can occur in retail transaction data, for example due to returns or adjustments. Net-sales summaries include these records unless a chart explicitly filters them.


## Exploratory Data Analysis

The following cells load the same Year 2009–2010 dataset used in Task 1, apply the same preparation rules for reproducibility, and then explore sales and customer patterns.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_online_retail_2009_2010.csv to cleaned_online_retail_2009_2010.csv


In [ ]:
import pandas as pd
import urllib.request

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"

urllib.request.urlretrieve(url, "/content/online_retail_II.xlsx")

df = pd.read_excel(
    "/content/online_retail_II.xlsx",
    sheet_name="Year 2009-2010"
)

# Repeat the same preparation here so the EDA section can run independently.
df = df.drop_duplicates()
df = df.dropna(subset=["Description"]).copy()
df["Customer ID"] = df["Customer ID"].fillna(0).astype("int64")
df["Country"] = df["Country"].str.strip()

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully!
Rows: 515668
Columns: 8


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 515668 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      515668 non-null  object        
 1   StockCode    515668 non-null  object        
 2   Description  515668 non-null  object        
 3   Quantity     515668 non-null  int64         
 4   InvoiceDate  515668 non-null  datetime64[ns]
 5   Price        515668 non-null  float64       
 6   Customer ID  515668 non-null  int64         
 7   Country      515668 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 35.4+ MB


In [ ]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,515668.000000,515668,515668.000000,515668.000000
mean,10.617719,2010-06-28 11:56:09.179162112,4.742955,12230.155400
min,-9360.000000,2009-12-01 07:45:00,-53594.360000,0.000000
25%,1.000000,2010-03-21 13:27:00,1.250000,12748.000000
50%,3.000000,2010-07-06 09:58:30,2.100000,14606.000000
75%,10.000000,2010-10-15 12:20:00,4.210000,16455.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,98.548503,NaN,147.507035,6360.284317


In [ ]:
df['Sales'] = df['Quantity'] * df['Price']

print("Total Net Sales:", round(df['Sales'].sum(), 2))
print("Average Net Sales per transaction line:", round(df['Sales'].mean(), 2))

Total Net Sales: 9505775.56
Average Net Sales per transaction line: 18.43


In [ ]:
country_sales = df.groupby('Country')['Sales'].sum()

country_sales = country_sales.sort_values(ascending=False)

print(country_sales.head(10))

Country
United Kingdom    8161692.763
EIRE               352235.680
Netherlands        263861.760
Germany            195920.421
France             130661.550
Sweden              51190.110
Denmark             46972.950
Switzerland         43343.410
Spain               37052.130
Australia           30051.800
Name: Sales, dtype: float64


In [ ]:

import plotly.express as px
top_countries = (
    country_sales.sort_values(ascending=False)
    .head(10)
    .reset_index())
top_countries.columns = ["Country", "Sales"]
top_countries = top_countries.sort_values("Sales")

# Light and attractive pastel colours
colors = [
    "#7CCBA2",  # Mint green
    "#82B1FF",  # Sky blue
    "#B39DDB",  # Lavender
    "#FFAB91",  # Peach
    "#F48FB1",  # Pink
    "#80DEEA",  # Aqua
    "#AED581",  # Light green
    "#FFD180",  # Soft orange
    "#90CAF9",  # Light blue
    "#CE93D8"   # Purple
]
fig = px.bar(
    top_countries,
    x="Sales",
    y="Country",
    orientation="h",
    color="Country",
    color_discrete_sequence=colors,
    text="Sales",
    title="Top 10 Countries by Net Sales",
    template="plotly_white",
    hover_data={"Sales": ":,.2f"}
)
fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    textfont=dict(size=16, color="#333333"),
    cliponaxis=False,
    marker_line_width=0
)
fig.update_layout(
    title=dict(
        text="🌍 Top 10 Countries by Net Sales",
        x=0.5,
        font=dict(size=28, color="#34495E") ),
    xaxis=dict(
        title="Net Sales",
        title_font=dict(size=18),
        tickfont=dict(size=14),
        tickformat=",.0f",
        gridcolor="#EAECEF"),
    yaxis=dict(
        title="Country",
        title_font=dict(size=18),
        tickfont=dict(size=16) ),
    height=500,
    width=900,
    showlegend=False,
    margin=dict(l=110, r=110, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="#FAFCFF",
    font=dict(family="Arial", size=15)
)
fig.show()


In [ ]:

import pandas as pd
import plotly.express as px

df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = df.groupby("Month")["Sales"].sum().reset_index()

fig = px.line(
    monthly_sales,
    x="Month",
    y="Sales",
    markers=True,
    title="Monthly Net Sales Trend",
    template="plotly_white"
)

fig.update_traces(
    line=dict(color="#7C83FD", width=4),
    marker=dict(size=10, color="#FF8FAB"),
    hovertemplate="Month: %{x}<br>Sales: %{y:,.2f}<extra></extra>"
)

fig.update_layout(
    title=dict(text="📈 Monthly Net Sales Trend", x=0.5, font=dict(size=24)),
    xaxis_title="Month",
    yaxis_title="Net Sales",
    height=450,
    width=850,
    font=dict(size=13),
    plot_bgcolor="white"
)

fig.show()


In [ ]:

highest_month = monthly_sales.loc[monthly_sales["Sales"].idxmax()]
lowest_month = monthly_sales.loc[monthly_sales["Sales"].idxmin()]

print("Highest Net Sales Month:", highest_month["Month"])
print("Highest Monthly Net Sales:", round(highest_month["Sales"], 2))

print("\nLowest Net Sales Month:", lowest_month["Month"])
print("Lowest Monthly Net Sales:", round(lowest_month["Sales"], 2))


Highest Net Sales Month: 2010-11
Highest Monthly Net Sales: 1416697.2

Lowest Net Sales Month: 2010-12
Lowest Monthly Net Sales: 376266.82


In [ ]:
# Exclude Customer ID = 0 because it represents a missing customer ID.
valid_customers = df[df["Customer ID"] != 0]
top_customers = (
    valid_customers.groupby("Country")["Customer ID"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Unique Customers")
)
print(top_customers)


          Country  Unique Customers
0  United Kingdom              4035
1         Germany                68
2          France                47
3           Spain                25
4     Netherlands                23
5        Portugal                18
6         Belgium                17
7          Sweden                16
8       Australia                15
9     Switzerland                14


In [ ]:

colors = [
    "#7CCBA2", "#82B1FF", "#B39DDB", "#FFAB91",
    "#F48FB1", "#80DEEA", "#AED581", "#FFD180",
    "#90CAF9", "#CE93D8"
]

fig = px.bar(
    top_customers.sort_values("Unique Customers"),
    x="Unique Customers",
    y="Country",
    orientation="h",
    color="Country",
    color_discrete_sequence=colors,
    text="Unique Customers",
    title="Top 10 Countries by Unique Customers",
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside",
    textfont=dict(size=14, color="#333333"),
    cliponaxis=False
)

fig.update_layout(
    title=dict(
        text="🌍 Top 10 Countries by Unique Customers",
        x=0.5,
        font=dict(size=24, color="#34495E")
    ),
    height=500,
    width=900,
    showlegend=False,
    margin=dict(l=110, r=100, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="#FAFCFF",
    font=dict(family="Arial", size=14)
)

fig.show()


In [ ]:

top_country = top_customers.loc[
    top_customers["Unique Customers"].idxmax()
]
print("Country with Most Customers:", top_country["Country"])
print("Number of Unique Customers:", top_country["Unique Customers"])


Country with Most Customers: United Kingdom
Number of Unique Customers: 4035


In [ ]:

top_products = (
    df[df["Quantity"] > 0]
    .groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
    .reset_index()
)

top_products.columns = ["Product", "Quantity Sold"]

print(top_products)


                              Product  Quantity Sold
0      STRAWBERRY CERAMIC TRINKET BOX          27060
1                SMALL POPCORN HOLDER          29809
2             JUMBO BAG RED RETROSPOT          30748
3  PACK OF 60 PINK PAISLEY CAKE CASES          31805
4         60 TEATIME FAIRY CAKE CASES          36348
5       ASSORTED COLOUR BIRD ORNAMENT          45236
6    PACK OF 72 RETRO SPOT CAKE CASES          46728
7                 BROCADE RING PURSE           48384
8   WORLD WAR 2 GLIDERS ASSTD DESIGNS          55058
9  WHITE HANGING HEART T-LIGHT HOLDER          58692


In [ ]:

import plotly.express as px

fig = px.bar(
    top_products.sort_values("Quantity Sold", ascending=True),
    x="Quantity Sold",
    y="Product",
    orientation="h",
    color="Quantity Sold",
    color_continuous_scale=[
        [0, "#E6D9FF"],
        [0.25, "#C4A7FF"],
        [0.5, "#9870E8"],
        [0.75, "#7041C5"],
        [1, "#452078"]
    ],
    text="Quantity Sold",
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    textfont=dict(size=10, color="#333333"),
    cliponaxis=False
)

fig.update_layout(
    title=dict(
        text="📦 Top 10 Products by Quantity Sold",
        x=0.5,
        font=dict(size=19, color="#452078")
    ),
    height=450,
    width=1000,
    bargap=0.3,
    coloraxis_showscale=False,
    margin=dict(l=10, r=65, t=65, b=55),
    xaxis=dict(
        title="Quantity Sold",
        gridcolor="#EEEEF5",
        zeroline=False,
        tickfont=dict(size=10)
    ),
    yaxis=dict(
        title="",
        tickfont=dict(size=9),
        automargin=True
    ),
    plot_bgcolor="white",
    paper_bgcolor="#FAF9FF",
    font=dict(family="Arial", size=10)
)

fig.show()


In [ ]:

df["Sales"] = df["Quantity"] * df["Price"]

sales_summary = df[(df["Quantity"] > 0) & (df["Price"] >= 0)][
    ["Quantity", "Sales"]
].describe()

print("Summary for positive-quantity rows with non-negative prices:")
print(sales_summary)


Summary for positive-quantity rows with non-negative prices:
            Quantity          Sales
count  505190.000000  505190.000000
mean       11.564956      20.333214
std        88.753421      91.471450
min         1.000000       0.000000
25%         1.000000       4.200000
50%         3.000000      10.200000
75%        12.000000      17.700000
max     19152.000000   25111.090000


In [ ]:

import plotly.express as px

valid_sales = df[(df["Quantity"] > 0) & (df["Price"] >= 0)]
sample_df = valid_sales.sample(
    n=min(3000, len(valid_sales)),
    random_state=42
).copy()

sample_df["Sales"] = sample_df["Quantity"] * sample_df["Price"]

fig = px.scatter(
    sample_df,
    x="Quantity",
    y="Sales",
    color="Sales",
    color_continuous_scale=["#C4A7FF", "#9870E8", "#452078"],
    hover_data=["Description", "Price"],
    title="Quantity vs Sales Analysis",
    template="plotly_white",
    opacity=0.7
)

fig.update_layout(
    title=dict(text="📊 Quantity vs Sales Analysis", x=0.5),
    height=550,
    width=950,
    xaxis_title="Quantity Purchased",
    yaxis_title="Sales Value",
    plot_bgcolor="white",
    paper_bgcolor="#FAF9FF"
)

fig.show()


In [ ]:

daily_sales = (
    df.groupby(df["InvoiceDate"].dt.date)["Sales"]
    .sum()
    .reset_index()
)

daily_sales.columns = ["Date", "Total Sales"]

fig = px.line(
    daily_sales,
    x="Date",
    y="Total Sales",
    title="Daily Net Sales Trend",
    template="plotly_white"
)

fig.update_traces(
    line=dict(color="#7041C5", width=2),
    hovertemplate="Date: %{x}<br>Sales: %{y:,.2f}<extra></extra>"
)

fig.update_layout(
    title=dict(text="📈 Daily Net Sales Trend", x=0.5),
    height=450,
    width=950,
    xaxis_title="Date",
    yaxis_title="Net Sales",
    plot_bgcolor="white",
    paper_bgcolor="#FAF9FF"
)

fig.show()


In [ ]:

# 1. Top country by sales
df["Sales"] = df["Quantity"] * df["Price"]

country_sales = (
    df.groupby("Country")["Sales"]
    .sum()
    .sort_values(ascending=False)
)
print("1. TOP COUNTRY BY NET SALES")
print(country_sales.head(1))

# 2. Highest sales month
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)
monthly_sales = df.groupby("Month")["Sales"].sum()
print("\n2. HIGHEST SALES MONTH")
print(monthly_sales.idxmax(), "-", round(monthly_sales.max(), 2))

# 3. Top country by unique customers (ignore missing-ID marker 0)
valid_customers = df[df["Customer ID"] != 0]
customers_by_country = (
    valid_customers.groupby("Country")["Customer ID"]
    .nunique()
    .sort_values(ascending=False)
)
print("\n3. TOP COUNTRY BY UNIQUE CUSTOMERS")
print(customers_by_country.head(1))

# 4. Top-selling product by quantity
top_product = (
    df[df["Quantity"] > 0]
    .groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
)
print("\n4. TOP-SELLING PRODUCT BY QUANTITY")
print(top_product.head(1))

# 5. Total calculated sales
print("\n5. TOTAL CALCULATED NET SALES")
print(round(df["Sales"].sum(), 2))


1. TOP COUNTRY BY NET SALES
Country
United Kingdom    8161692.763
Name: Sales, dtype: float64

2. HIGHEST SALES MONTH
2010-11 - 1416697.2

3. TOP COUNTRY BY UNIQUE CUSTOMERS
Country
United Kingdom    4035
Name: Customer ID, dtype: int64

4. TOP-SELLING PRODUCT BY QUANTITY
Description
WHITE HANGING HEART T-LIGHT HOLDER    58692
Name: Quantity, dtype: int64

5. TOTAL CALCULATED NET SALES
9505775.56


## Key Findings and Conclusion

- The United Kingdom has the highest net sales in this dataset.
- November 2010 is the highest net-sales month in the current analysis; December 2010 is the lowest, and the selected sheet ends during December.
- The United Kingdom has the largest number of identified customers when Customer ID `0` is excluded.
- `WHITE HANGING HEART T-LIGHT HOLDER` is the top-selling product by positive quantity in the current analysis.
- Negative quantities and prices can affect net-sales totals, so results should be interpreted with transaction returns and adjustments in mind.

**Conclusion:** EDA helps highlight the main sales markets, monthly trends, popular products, and customer distribution in the selected Online Retail II data.
